# SLIDE Workflow

This notebook mirrors the conceptual workflow in Figure 1 of the paper: infer a ruggedness parameter from mutation-only decay, use that value to place the landscape in a directed-evolution strategy space, and compare baseline, SLIDE-selected, and high-exploration strategies.


## Setup

Load the NK simulation, decay fitting, and plotting tools used by the workflow. The notebook writes only a small workflow-specific strategy-space cache into `raw_data/`, so repeated runs can skip the 7-by-7 sweep.


In [ ]:
%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from slide import selection_function_library as slct
from slide.data_generation import repeated_population, strategy_grid
from slide.direvo_functions import (
    build_NK_landscape_function,
    build_mutation_function,
    build_selection_function,
    get_single_decay_rate,
    model_function,
    run_diffusion,
    run_directed_evolution,
)
from slide.utils import get_figures_dir, get_raw_data_dir, load_pickle, parameterized_filename, save_pickle

RAW_DATA_DIR = get_raw_data_dir()
FIGURES_DIR = get_figures_dir()
for figure_type in ("pdf", "png", "eps"):
    (FIGURES_DIR / figure_type).mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "DejaVu Sans"
print(f"raw_data: {RAW_DATA_DIR}")
print(f"figures: {FIGURES_DIR}")


## Workflow Parameters

The decay step uses the Figure 3 mutation-rate convention for NK ruggedness inference. The strategy step uses the Figure 5 NK strategy-space convention: total mutation rate `0.1`, population size `1200`, `25` generations, and a 7-by-7 base-chance/splitting grid.


In [ ]:
# NK landscape and decay acquisition.
N = 50
K = 25
A = 2
RHO_NK = (K + 1) / N
LANDSCAPE_SEED = 25
START_SEED = 26
START_CANDIDATES = 512

DIFFUSION_MUTATION_RATE = 0.5
DIFFUSION_POPSIZE = 2500
DIFFUSION_REPS = 3
DIFFUSION_STEPS = 25
DIFFUSION_SEED = 101

# Directed-evolution strategy space.
STRATEGY_MUTATION_RATE = 0.1
STRATEGY_POPSIZE = 1200
STRATEGY_STEPS = 25
STRATEGY_GRID_SIZE = 7
STRATEGY_REPS = 5
STRATEGY_SEED = 303
OVERWRITE_STRATEGY_SPACE = False

# Strategy comparison traces.
DE_TRACE_REPS = 8
DE_TRACE_SEED = 404
SAVE_FIGURES = True

workflow_strategy_filename = parameterized_filename(
    "slide_workflow_nk_strategy",
    N=N,
    A=A,
    K=K,
    mu=STRATEGY_MUTATION_RATE,
    pop=STRATEGY_POPSIZE,
    reps=STRATEGY_REPS,
    steps=STRATEGY_STEPS,
    grid=STRATEGY_GRID_SIZE,
    seed=STRATEGY_SEED,
)
workflow_strategy_path = RAW_DATA_DIR / workflow_strategy_filename
workflow_strategy_path


## 1. Decay Curve Acquisition

Instantiate a reproducible `N=50`, `K=25`, `A=2` NK landscape and choose one high-fitness starting genotype. In an experimental workflow this would correspond to the starting genotype available before the exploratory mutation-only phase.


In [ ]:
landscape_key = jr.PRNGKey(LANDSCAPE_SEED)
fitness_function = build_NK_landscape_function(landscape_key, N, K)

candidate_key = jr.PRNGKey(START_SEED)
candidate_starts = jr.randint(candidate_key, (START_CANDIDATES, N), 0, A)
candidate_fitness = np.asarray(fitness_function(candidate_starts))
start_index = int(np.argmax(candidate_fitness))
start = np.asarray(candidate_starts[start_index], dtype=np.int32)
start_fitness = float(candidate_fitness[start_index])

print(f"rho_NK = {RHO_NK:.3f}")
print(f"Selected start index: {start_index}")
print(f"Selected start fitness: {start_fitness:.3f}")


Run three independent mutation-only diffusion experiments from the same starting genotype. The population is not selected or resampled during this step; it only mutates, matching the decay-rate inference assumption.


In [ ]:
diffusion_mutation_function = build_mutation_function(DIFFUSION_MUTATION_RATE / N, A)
diffusion_initial_population = repeated_population(start, DIFFUSION_POPSIZE)
diffusion_keys = jr.split(jr.PRNGKey(DIFFUSION_SEED), DIFFUSION_REPS)

diffusion_histories = jax.jit(
    jax.vmap(
        lambda key: run_diffusion(
            key,
            diffusion_initial_population,
            diffusion_mutation_function,
            fitness_function=fitness_function,
            num_steps=DIFFUSION_STEPS,
        )[1]
    )
)(diffusion_keys)

fitness_history = np.asarray(diffusion_histories["fitness"])
F_mu_replicates = fitness_history.mean(axis=-1)
F_mu = F_mu_replicates.mean(axis=0)
G_mu_replicates = F_mu_replicates**2
G_mu = G_mu_replicates.mean(axis=0)

generations = np.arange(DIFFUSION_STEPS)
print(f"fitness_history shape: {fitness_history.shape}")
print(f"F_mu shape: {F_mu.shape}")
print(f"G_mu shape: {G_mu.shape}")


## 2. Rho Calculation

Fit the mean-fitness decay rate `rho` from `F_mu` and the squared-decay rate `rho_2` from `G_mu`. The squared curve is fitted with the same exponential model and divided by two, matching the convention used for Figure 4 empirical-landscape analyses.


In [ ]:
F_mu_norm = F_mu / F_mu[0]
G_mu_norm = G_mu / G_mu[0]
G_mu_replicates_norm = G_mu_replicates / G_mu_replicates[:, [0]]

rho_fit, F_constant = get_single_decay_rate(
    F_mu_norm,
    mut=DIFFUSION_MUTATION_RATE,
    num_steps=DIFFUSION_STEPS,
)
rho_2_raw, G_constant = get_single_decay_rate(
    G_mu_norm,
    mut=DIFFUSION_MUTATION_RATE,
    num_steps=DIFFUSION_STEPS,
)
rho_2_fit = rho_2_raw / 2

F_mu_fit = model_function(generations, rho_fit, F_constant, mut=DIFFUSION_MUTATION_RATE)
G_mu_fit = model_function(generations, rho_2_raw, G_constant, mut=DIFFUSION_MUTATION_RATE)

print(f"rho_NK = {RHO_NK:.3f}")
print(f"rho_fit from F_mu = {rho_fit:.3f}")
print(f"rho_2 from G_mu = {rho_2_fit:.3f}")


## 3. Visualise Decay Fits

Plot the observed mutation-only decay curves and their fitted exponentials. The panel titles compare the analytical NK ruggedness `rho_NK` with the fitted `rho` and `rho_2` estimates.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.8), dpi=300, constrained_layout=True)

for curve in F_mu_replicates / F_mu_replicates[:, [0]]:
    axes[0].plot(generations, curve, color="0.75", linewidth=0.8, alpha=0.8)
axes[0].scatter(generations, F_mu_norm, s=18, color="tab:blue", label=r"Observed $F_\mu$")
axes[0].plot(generations, F_mu_fit, color="black", linewidth=1.3, label="Exponential fit")
axes[0].set_title(rf"$F_\mu$: $\rho_{{NK}}={RHO_NK:.2f}$, $\rho={rho_fit:.2f}$", fontsize=10)
axes[0].set_xlabel("Generations")
axes[0].set_ylabel(r"Normalized $F_\mu$")
axes[0].legend(fontsize=7, frameon=False)
axes[0].grid(alpha=0.25)

for curve in G_mu_replicates_norm:
    axes[1].plot(generations, curve, color="0.75", linewidth=0.8, alpha=0.8)
axes[1].scatter(generations, G_mu_norm, s=18, color="tab:orange", label=r"Observed $G_\mu$")
axes[1].plot(generations, G_mu_fit, color="black", linewidth=1.3, label="Exponential fit")
axes[1].set_title(rf"$G_\mu$: $\rho_{{NK}}={RHO_NK:.2f}$, $\rho_2={rho_2_fit:.2f}$", fontsize=10)
axes[1].set_xlabel("Generations")
axes[1].set_ylabel(r"Normalized $G_\mu$")
axes[1].legend(fontsize=7, frameon=False)
axes[1].grid(alpha=0.25)

if SAVE_FIGURES:
    for figure_type in ("pdf", "png", "eps"):
        fig.savefig(FIGURES_DIR / figure_type / f"slide_workflow_decay.{figure_type}", dpi=300)
plt.show()


## 4. NK Strategy Space

Evaluate the Figure 5 NK strategy grid on this same landscape, or load the workflow-specific raw pickle if it already exists. Rows are population splitting values and columns are base-chance values. The threshold paired with each base chance is chosen to keep the selected fraction fixed, as in the paper strategy sweeps.


In [ ]:
thresholds, base_chances, splits = strategy_grid(STRATEGY_GRID_SIZE)
thresholds = np.asarray(thresholds, dtype=float)
base_chances = np.asarray(base_chances, dtype=float)
splits = np.asarray(splits, dtype=int)
strategy_mutation_function = build_mutation_function(STRATEGY_MUTATION_RATE / N, A)


def run_de_split_history(rng, split_size, base_chance, threshold):
    selection_params = {"threshold": float(threshold), "base_chance": float(base_chance)}
    selection_function = build_selection_function(slct.base_chance_threshold_select, selection_params)
    subpopsize = STRATEGY_POPSIZE // int(split_size)
    initial_subpopulation = repeated_population(start, subpopsize)
    return jax.jit(
        jax.vmap(
            lambda key: run_directed_evolution(
                key,
                initial_subpopulation,
                selection_function,
                strategy_mutation_function,
                fitness_function=fitness_function,
                num_steps=STRATEGY_STEPS,
            )[1]
        )
    )(jr.split(rng, int(split_size)))


def run_strategy_final_score(rng, split_size, base_chance, threshold):
    history = run_de_split_history(rng, split_size, base_chance, threshold)
    return float(np.asarray(history["fitness"])[:, :, -1].max())


In [ ]:
if workflow_strategy_path.exists() and not OVERWRITE_STRATEGY_SPACE:
    strategy_payload = load_pickle(workflow_strategy_path)
    strategy_scores = np.asarray(strategy_payload["data"])
    print(f"Loaded strategy space from {workflow_strategy_path}")
else:
    strategy_scores = np.zeros((len(splits), len(base_chances), STRATEGY_REPS), dtype=float)
    strategy_keys = iter(jr.split(jr.PRNGKey(STRATEGY_SEED), int(strategy_scores.size)))
    for split_index, split_size in enumerate(tqdm(splits, desc="Strategy split rows")):
        for base_index, (base_chance, threshold) in enumerate(zip(base_chances, thresholds)):
            for rep_index in range(STRATEGY_REPS):
                strategy_scores[split_index, base_index, rep_index] = run_strategy_final_score(
                    next(strategy_keys),
                    int(split_size),
                    float(base_chance),
                    float(threshold),
                )
    strategy_payload = {
        "data": strategy_scores,
        "params": {
            "N": N,
            "K": K,
            "A": A,
            "rho_NK": RHO_NK,
            "mutation_rate": STRATEGY_MUTATION_RATE,
            "popsize": STRATEGY_POPSIZE,
            "M": STRATEGY_STEPS,
            "strategy_grid_size": STRATEGY_GRID_SIZE,
            "num_reps": STRATEGY_REPS,
            "thresholds": thresholds,
            "base_chances": base_chances,
            "splits": splits,
            "landscape_seed": LANDSCAPE_SEED,
            "start": start,
            "seed": STRATEGY_SEED,
        },
        "metadata": {
            "description": "Workflow NK strategy-space sweep for the Figure 1 SLIDE schematic.",
            "paper_reference": "Figure 1 and Figure 5 strategy-grid convention.",
            "filename": workflow_strategy_filename,
        },
    }
    save_pickle(strategy_payload, workflow_strategy_path)
    print(f"Saved strategy space to {workflow_strategy_path}")

strategy_mean = strategy_scores.mean(axis=-1)
optimal_split_index, optimal_base_index = np.unravel_index(np.argmax(strategy_mean), strategy_mean.shape)

baseline_split_index = int(np.where(splits == 1)[0][0])
baseline_base_index = 0
high_exploration_split_index = int(np.argmax(splits))
high_exploration_base_index = int(np.argmax(base_chances))

strategy_choices = {
    "Baseline": {
        "split_index": baseline_split_index,
        "base_index": baseline_base_index,
        "color": "tab:blue",
    },
    "Optimal": {
        "split_index": int(optimal_split_index),
        "base_index": int(optimal_base_index),
        "color": "tab:red",
    },
    "High exploration": {
        "split_index": high_exploration_split_index,
        "base_index": high_exploration_base_index,
        "color": "tab:green",
    },
}
for choice in strategy_choices.values():
    choice["split"] = int(splits[choice["split_index"]])
    choice["base_chance"] = float(base_chances[choice["base_index"]])
    choice["threshold"] = float(thresholds[choice["base_index"]])

strategy_choices


## 5. Directed Evolution With Three Strategies

Run directed evolution from the same starting genotype with three parameter choices: a baseline strategy, the optimal strategy from the measured strategy space, and a deliberately high-exploration strategy with maximum splitting and maximum base chance.


In [ ]:
de_traces = {}
for offset, (name, choice) in enumerate(strategy_choices.items()):
    trace_keys = jr.split(jr.PRNGKey(DE_TRACE_SEED + offset), DE_TRACE_REPS)
    traces = []
    for key in tqdm(trace_keys, desc=f"DE traces: {name}"):
        history = run_de_split_history(
            key,
            choice["split"],
            choice["base_chance"],
            choice["threshold"],
        )
        fitness = np.asarray(history["fitness"])
        traces.append(fitness.max(axis=(0, 2)))
    de_traces[name] = np.asarray(traces)

{key: value.shape for key, value in de_traces.items()}


Visualise both the directed-evolution traces and the corresponding strategy-space map. The right panel follows the Figure 5 strategy-space layout: base chance on the x-axis and number of subpopulations on the y-axis.


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(7.2, 3.0),
    dpi=300,
    constrained_layout=True,
    gridspec_kw={"width_ratios": [1.35, 1.0]},
)

trace_ax, space_ax = axes
for name, traces in de_traces.items():
    color = strategy_choices[name]["color"]
    mean_trace = traces.mean(axis=0)
    std_trace = traces.std(axis=0)
    label = f"{name} (split={strategy_choices[name]['split']}, b={strategy_choices[name]['base_chance']:.2f})"
    strategy_generations = np.arange(STRATEGY_STEPS)
    trace_ax.plot(strategy_generations, mean_trace, color=color, linewidth=1.4, label=label)
    trace_ax.fill_between(strategy_generations, mean_trace - std_trace, mean_trace + std_trace, color=color, alpha=0.18)
trace_ax.set_title("Directed-evolution outcomes", fontsize=10)
trace_ax.set_xlabel("Generations")
trace_ax.set_ylabel("Best fitness observed")
trace_ax.legend(fontsize=6, frameon=False)
trace_ax.grid(alpha=0.25)

image = space_ax.imshow(strategy_mean, aspect="auto", origin="upper", cmap="viridis")
for name, choice in strategy_choices.items():
    marker = "*" if name == "Optimal" else "o"
    size = 95 if name == "Optimal" else 55
    space_ax.scatter(
        choice["base_index"],
        choice["split_index"],
        s=size,
        marker=marker,
        color=choice["color"],
        edgecolor="white",
        linewidth=0.7,
        label=name,
    )
space_ax.set_title("Strategy space", fontsize=10)
space_ax.set_xlabel("Base chance")
space_ax.set_ylabel("Subpopulations")
space_ax.set_xticks([0, len(base_chances) - 1])
space_ax.set_xticklabels([f"{base_chances[0]:.2f}", f"{base_chances[-1]:.2f}"])
space_ax.set_yticks([0, len(splits) - 1])
space_ax.set_yticklabels([str(splits[0]), str(splits[-1])])
space_ax.tick_params(labelsize=7)
space_ax.legend(fontsize=6, loc="center left", bbox_to_anchor=(1.05, 0.5), frameon=False)
cbar = fig.colorbar(image, ax=space_ax, fraction=0.046, pad=0.04)
cbar.set_label("Final fitness", fontsize=7)
cbar.ax.tick_params(labelsize=6)

fig.suptitle(rf"SLIDE workflow on NK landscape: $N={N}$, $K={K}$, $\rho_{{NK}}={RHO_NK:.2f}$", fontsize=11)

if SAVE_FIGURES:
    for figure_type in ("pdf", "png", "eps"):
        fig.savefig(FIGURES_DIR / figure_type / f"slide_workflow_strategy.{figure_type}", dpi=300, bbox_inches="tight")
plt.show()
